# LanceDB — User Journey

Unlike the others, LanceDB uses its own columnar format. This involves an explicit **convert** step: data is copied out of Parquet into Lance (written to S3), after which it lives outside LaminDB's lineage graph.

**Steps:** convert → query → filtered query → append → time travel

In [ ]:
import time
from contextlib import contextmanager

timings = {}

@contextmanager
def bench(step):
    """Time a step and record it for the final benchmark table."""
    t0 = time.perf_counter()
    yield
    timings[step] = time.perf_counter() - t0
    print(f"  {step}: {timings[step]:.3f}s")

## 1. Convert to LanceDB

`collection.open()` gives a PyArrow table directly; `create_table` writes it into Lance format on S3. This is the one engine where the data no longer lives in Parquet.

In [ ]:
import lamindb as ln
import lancedb
import pyarrow as pa
import pandas as pd

ln.track(path="lancedb_pipeline.ipynb", project="Lakehouse benchmarks v1")
collection = ln.Collection.get("K6X8Ejk3fjgAZT6h0000")

with bench("load_data"):
    arrow_table = collection.open().to_table()

db = lancedb.connect("lancedb_warehouse")

with bench("convert"):
    table = db.create_table("cnv_vcf", data=arrow_table, mode="overwrite")

print(f"Total rows: {table.count_rows():,}")


## 2. Query — per-sample stats and recurrent regions

In [ ]:
def calculate_sample_stats(arrow_table):
    """Summary statistics per sample."""
    df = arrow_table.to_pandas()
    rows = []
    for name in df["SAMPLE_NAME"].unique():
        s = df[df["SAMPLE_NAME"] == name]
        dels = s[s["INFO_SVLEN"] < 0]
        rows.append({
            "Sample": name,
            "Total_CNVs": len(s),
            "Deletions": len(dels),
            "Median_Deletion_Size": abs(dels["INFO_SVLEN"].median()) if not dels.empty else 0,
            "Homozygous_CNVs": (s["SAMPLE_GT"] == "1/1").sum(),
            "Heterozygous_CNVs": (s["SAMPLE_GT"] == "0/1").sum(),
        })
    return pd.DataFrame(rows)

def identify_recurrent_regions(arrow_table, proximity=1000):
    """Regions with recurrent CNVs across >=2 samples."""
    df = arrow_table.select(["CHROM", "POS", "SAMPLE_NAME"]).to_pandas()
    df["region_key"] = df["CHROM"] + ":" + ((df["POS"] // proximity) * proximity).astype(str)
    counts = df.groupby("region_key")["SAMPLE_NAME"].nunique()
    return counts[counts >= 2]

In [ ]:
with bench("query_stats"):
    stats_df = calculate_sample_stats(table.to_arrow())
stats_df.head()

In [ ]:
with bench("query_recurrent"):
    recurrent = identify_recurrent_regions(table.to_arrow())
print(f"Identified {len(recurrent)} recurrent regions.")

## 3. Filtered query — pushdown via the Lance scanner

In [ ]:
with bench("filtered_query"):
    filtered = table.to_lance().to_table(
        filter="CHROM = '1' AND POS BETWEEN 1000000 AND 50000000"
    ).to_pandas()
print(f"Variants in chr1:1M-50M: {len(filtered)}")

## 4. Append — every `add()` creates a new version automatically

In [ ]:
with bench("append"):
    table.add(arrow_table.slice(0, 10))  # placeholder — replace with real new data
print(f"Version: {table.version} | Total rows: {table.count_rows():,}")

## 5. Time travel — check out any previous version

In [ ]:
with bench("time_travel"):
    historical = db.open_table("cnv_vcf")
    historical.checkout(1)
print(f"Rows at v1: {historical.count_rows():,}")

## Benchmark summary

In [ ]:
import pandas as pd
pd.DataFrame(
    [{"step": k, "seconds": round(v, 3)} for k, v in timings.items()]
)

In [ ]:
try:
    ln.finish()
except Exception:
    pass
